In [ ]:
import pandas as pd
from ggner_3d.boundaries import make_tads, assign_domain_score
from ggner_3d.cm import get_expected_cis_df, prepare_view_df, CLR_CONNS
from ggner_3d import plotting
from ggner_3d.plotting import plot_flanks_start_end_stackup
plotting.update_rcparams()

from coolpuppy import coolpup
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import cooltools

window_bp = 500_000
bin_bp = 10_000
tolerance_bp = 10_000
clr_ = CLR_CONNS(resolution=bin_bp)
NPROC = 6

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
wtnouv_insulation = pd.read_csv('/home/carlos/Clone/ggner-3d/data/insulation/insulation_WTnoUV_10000bp.csv')
wtnouv_boundaries = wtnouv_insulation.loc[wtnouv_insulation[f'is_boundary_{window_bp}'] == True]
wtnouv_tads = make_tads(wtnouv_boundaries[['chrom', 'start', 'end']], maxlen=3e6)

domain_score_wtnouv = assign_domain_score(
    tads_df=wtnouv_tads,
    clr=clr_['WTnoUV'],
    expected=get_expected_cis_df('WTnoUV', resolution_kb=bin_bp // 1000, label='chrom'),
    view_df=prepare_view_df(arm=False),
    resolution=bin_bp,
    nproc=NPROC,
    clr_weight_name='weight',
)['domain_score']

domain_score_wt3h = assign_domain_score(
    tads_df=wtnouv_tads,
    clr=clr_['WT3h'],
    expected=get_expected_cis_df('WT3h', resolution_kb=bin_bp // 1000, label='chrom'),
    view_df=prepare_view_df(arm=False),
    resolution=bin_bp,
    nproc=NPROC,
    clr_weight_name='weight',
)['domain_score']

In [ ]:
wtnouv_tads.sort_values(['chrom', 'start'], inplace=True)
wtnouv_tads

In [ ]:
wtnouv_tads['domain_score_wtnouv'] = domain_score_wtnouv
wtnouv_tads['domain_score_wt3h'] = domain_score_wt3h

wtnouv_tads['Q'] = pd.qcut(domain_score_wtnouv, 4, labels=False) + 1

wtnouv_tads['Q_diff'] = pd.qcut(domain_score_wt3h - domain_score_wtnouv, 4, labels=False) + 1

tad_str = wtnouv_tads.copy()

In [ ]:
wt_nouv_dots = pd.read_csv('/home/carlos/Clone/ggner-3d/data/dots/dots_WTnoUV_10000.csv')

def prepare_strength(df):
    df = df.copy()
    df["fe_donut"] = df["count"] / df["la_exp.donut.value"]
    df["fe_vertical"] = df["count"] / df["la_exp.vertical.value"]
    df["fe_horizontal"] = df["count"] / df["la_exp.horizontal.value"]
    df["fe_lowleft"] = df["count"] / df["la_exp.lowleft.value"]

    df["fe_min"] = df[["fe_donut","fe_vertical","fe_horizontal","fe_lowleft"]].min(axis=1)
    df['Q'] = pd.qcut(df['fe_min'], 4, labels=False) + 1
    return df[['chrom1', 'start1', 'end2', 'fe_donut','fe_vertical','fe_horizontal','fe_lowleft','fe_min', 'Q']]

dots = prepare_strength(wt_nouv_dots)
dots = dots.rename(columns={'chrom1': 'chrom', 'start1':'start', 'end2':'end'})

In [ ]:
from ggner_3d.bbi_helpers import bbi_stackup_multiview, groupwise_delta, split_array_by_group
from scipy.ndimage import gaussian_filter1d

def gaussian_smooth(y, sigma=3, axis=0):
    return gaussian_filter1d(y, sigma=sigma, axis=axis) if sigma is not None else y

In [ ]:
ctcf_wt_dict = {
'wt_nouv_ctcf': [
    '/home/carlos/oldies/ner_collab/ctcf/bigwigs_rpgc/WT_NoUV_REP1.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/ctcf/bigwigs_rpgc/WT_NoUV_REP2.mLb.clN.rpgc.bw'
],
'wt_3h_ctcf': [
    '/home/carlos/oldies/ner_collab/ctcf/bigwigs_rpgc/WT_3h_REP1.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/ctcf/bigwigs_rpgc/WT_3h_REP2.mLb.clN.rpgc.bw'
]
}

# ds_wt_dict = {
# 'wt_nouv_ds': [
#     '/home/carlos/oldies/ner_collab/damageseq/U2OS_CPD_0h_DS_hg38_pe_sortedbyCoordinates.rpgc.bw'
# ],
# 'wt_3h_ds': [
#     '/home/carlos/oldies/ner_collab/damageseq/U2OS_CPD_3h_DS_hg38_pe_sortedbyCoordinates.rpgc.bw'
# ]
# }

ds_dict_obs = {
    'wt_nouv_ds': ['/home/carlos/oldies/ner_collab/damageseq/obs0_rpm.1000bp.bw'],
    'wt_3h_ds': ['/home/carlos/oldies/ner_collab/damageseq/obs3_rpm.1000bp.bw'],
}

ds_dict_exp = {
    'wt_nouv_ds': ['/home/carlos/oldies/ner_collab/damageseq/exp0_rpm.1000bp.bw'],
    'wt_3h_ds': ['/home/carlos/oldies/ner_collab/damageseq/exp3_rpm.1000bp.bw'],
}

k4me3_wt_dict = {
'3h_nouv_diff': ['/home/carlos/oldies/ner_collab/k4m3-ash-1h3h/3h.ASH1L_specific.delta_mean.bw'],
'1h_nouv_diff': ['/home/carlos/oldies/ner_collab/k4m3-ash-1h3h/1h.ASH1L_specific.delta_mean.bw']
}

xpc_wt_dict = {
'wt_nouv_xpc': [
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_noUV_REP1.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_noUV_REP2.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_noUV_REP3.mLb.clN.rpgc.bw'
],
'wt_3h_xpc': [
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_3h_REP1.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_3h_REP2.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_3h_REP3.mLb.clN.rpgc.bw'
],
'wt_1h_xpc': [
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_1h_REP1.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_1h_REP2.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_1h_REP3.mLb.clN.rpgc.bw'
]
}

ash1l_wt_al_dict = {
    'wt_nouv': ['/home/carlos/oldies/ner_collab/ASH1L_RPGC/AL_noUV_1.rpgc.bw',
                '/home/carlos/oldies/ner_collab/ASH1L_RPGC/AL_noUV_2.rpgc.bw',
                '/home/carlos/oldies/ner_collab/ASH1L_RPGC/AL_noUV_3.rpgc.bw'],
    'wt_3h': ['/home/carlos/oldies/ner_collab/ASH1L_RPGC/AL_3H_1.rpgc.bw',
              '/home/carlos/oldies/ner_collab/ASH1L_RPGC/AL_3H_2.rpgc.bw',
              '/home/carlos/oldies/ner_collab/ASH1L_RPGC/AL_3H_3.rpgc.bw']
}

ash1l_wt_ev_dict = {
    'wt_nouv': ['/home/carlos/oldies/ner_collab/ASH1L_RPGC/EV_noUV_1.rpgc.bw',
                '/home/carlos/oldies/ner_collab/ASH1L_RPGC/EV_noUV_2.rpgc.bw',
                '/home/carlos/oldies/ner_collab/ASH1L_RPGC/EV_noUV_3.rpgc.bw'],
    'wt_3h': ['/home/carlos/oldies/ner_collab/ASH1L_RPGC/EV_3H_1.rpgc.bw',
              '/home/carlos/oldies/ner_collab/ASH1L_RPGC/EV_3H_2.rpgc.bw',
              '/home/carlos/oldies/ner_collab/ASH1L_RPGC/EV_3H_3.rpgc.bw']
}

ash1l_wt_al_sub_ev_dict = {
    'wt_nouv': ['/home/carlos/oldies/ner_collab/ASH1L_RPGC/ALminusEV_noUV.bw'],
    'wt_3h': ['/home/carlos/oldies/ner_collab/ASH1L_RPGC/ALminusEV_3H.bw']
}

# atac_wt_dict = {
#     'wt_nouv_atac': [
#         '/home/carlos/oldies/ner_collab/atac/WT_noUV_1.fc.bigwig',
#         '/home/carlos/oldies/ner_collab/atac/WT_noUV_2.fc.bigwig'
#     ],
#     'wt_3h_atac': [
#         '/home/carlos/oldies/ner_collab/atac/WT_3H_1.fc.bigwig',
#         '/home/carlos/oldies/ner_collab/atac/WT_3H_2.fc.bigwig'
#     ]
# }

atac_wt_dict = {
    'wt_nouv_atac': [
        '/home/carlos/oldies/ner_collab/atac-rpgc-bws/WT_noUV_1.rpgc.bw',
        '/home/carlos/oldies/ner_collab/atac-rpgc-bws/WT_noUV_2.rpgc.bw'
    ],
    'wt_3h_atac': [
        '/home/carlos/oldies/ner_collab/atac-rpgc-bws/WT_3H_1.rpgc.bw',
        '/home/carlos/oldies/ner_collab/atac-rpgc-bws/WT_3H_2.rpgc.bw'
    ]
}

In [ ]:
ctcf_original = {
    'wt_nouv_ctcf':
    [
        '/home/carlos/oldies/ner_collab/ctcf/bigwigs/WT_NoUV_REP1.mLb.clN.bigWig',
        '/home/carlos/oldies/ner_collab/ctcf/bigwigs/WT_NoUV_REP2.mLb.clN.bigWig'
    ],
    'wt_3h_ctcf':
    [
        '/home/carlos/oldies/ner_collab/ctcf/bigwigs/WT_3h_REP1.mLb.clN.bigWig',
        '/home/carlos/oldies/ner_collab/ctcf/bigwigs/WT_3h_REP2.mLb.clN.bigWig'
    ]
}

In [ ]:
from __future__ import annotations

from typing import Dict, Mapping, Tuple, Any

# ----------------------------
# Shared config / constants
# ----------------------------
GROUP_COL = "Q"
GROUPS: Tuple[int, ...] = (1, 2, 3, 4)
flank_mode="proportional"
n_main = 100
flank_frac = 0.25
flank_bp=100_000
n_flank = 25
sigma = None

STACKUP_KWARGS = dict(
    nbins_main=n_main,
    nbins_flank=n_flank,
    flank_mode=flank_mode,
    flank_bp=flank_bp,
    flank_frac=flank_frac,
    flank_min_bp=None,
    flank_max_bp=None,
    summary_per_stack="mean",
    summary_across_stacks="mean",
)

# ----------------------------
# Helpers
# ----------------------------
def run_stackup(bigwig_paths: Mapping[str, list[str]], regions) -> Dict[str, Any]:
    """Run stackup for a multiview bigwig dict."""
    return bbi_stackup_multiview(bigwig_paths=bigwig_paths, regions=regions, **STACKUP_KWARGS)

def smooth_views(view_arrays: Mapping[str, Any], sigma: float, axis1_trim_cutoffs=(0.05, 0.95)) -> Dict[str, Any]:
    return {k: gaussian_smooth(v, sigma=sigma) for k, v in view_arrays.items()} if sigma is not None else view_arrays

def delta_by_group(treated: Any, control: Any, df: Any) -> Any:
    """Compute groupwise delta (treated - control)."""
    return groupwise_delta(
        df,
        treated,
        control,
        group_col=GROUP_COL,
        groups=GROUPS,
    )

def split_by_group(arr: Any, df: Any, group_col: str = GROUP_COL) -> Any:
    """Split an array by group."""
    return split_array_by_group(
        df,
        arr,
        group_col=group_col,
        groups=GROUPS,
    )

# ----------------------------
# Assay definitions
# ----------------------------
assays = {

    "ctcf_original": {
        "paths": ctcf_original,
    },
    # "atac": {
    #     "paths": atac_wt_dict,
    #     "delta": ("wt_3h_atac", "wt_nouv_atac"),
    #     "split": ("wt_nouv_atac",)
    # },
    # "ctcf": {
    #     "paths": ctcf_wt_dict,
    #     "delta": ("wt_3h_ctcf", "wt_nouv_ctcf"),
    # },
    # "ds": {
    #     "paths": ds_dict_obs,
    #     "delta": ("wt_nouv_ds", "wt_3h_ds"),
    #     "split": ("wt_nouv_ds",),
    # },
    # "k4me3_3h": {
    #     "paths": k4me3_wt_dict,
    #     "split": ("3h_nouv_diff",),
    # },
    # "k4me3_1h": {
    #     "paths": k4me3_wt_dict,
    #     "split": ("1h_nouv_diff",),
    # },
    # "xpc_3h": {
    #     "paths": xpc_wt_dict,
    #     "delta": ("wt_3h_xpc", "wt_nouv_xpc"),
    # },
    # "xpc_1h": {
    #     "paths": xpc_wt_dict,
    #     "delta": ("wt_1h_xpc", "wt_nouv_xpc"),
    # },
    # "ash1l_al_sub_ev": {
    #     "paths": ash1l_wt_al_sub_ev_dict,
    #     "delta": ("wt_3h", "wt_nouv"),
    # },
    # "ash1l_al": {"paths": ash1l_wt_al_dict, "delta": ("wt_3h", "wt_nouv")},
    # "ash1l_ev": {"paths": ash1l_wt_ev_dict, "delta": ("wt_3h", "wt_nouv")},

#     "xpc":
#         {
#             "paths": xpc_wt_dict,
# },

#     'atac':
#         {
#             "paths": atac_wt_dict,
#         },
}

# ----------------------------
# Run loops
# ----------------------------

# regions_loops = dots.copy()#.sample(n=500)
# results_loops: Dict[str, Dict[str, Any]] = {}
# for name, cfg in assays.items():
#     print(f"Processing assay: {name}, using {regions_loops.shape[0]} loops")
#     raw = run_stackup(cfg["paths"], regions=regions_loops[["chrom", "start", "end"]])
#     sm = smooth_views(raw, sigma=sigma)

#     out: Dict[str, Any] = {"raw": raw, "smoothed": sm}

#     if "delta" in cfg:
#         treat_key, ctrl_key = cfg["delta"]
#         out["delta_by_group"] = delta_by_group(sm[treat_key], sm[ctrl_key], df=regions_loops)

#     if "split" in cfg:
#         (key,) = cfg["split"]
#         out["split_by_group"] = split_by_group(sm[key], df=regions_loops)

#     results_loops[name] = out

# ----------------------------
# Run tads
# ----------------------------
results_tads: Dict[str, Dict[str, Any]] = {}
regions_tads = tad_str.copy()#.sample(n=500)

for name, cfg in assays.items():
    print(f"Processing assay: {name}, using {regions_tads.shape[0]} TADs")
    raw = run_stackup(cfg["paths"], regions=regions_tads[["chrom", "start", "end"]])
    sm = smooth_views(raw, sigma=sigma)

    out: Dict[str, Any] = {"raw": raw, "smoothed": sm}

    if "delta" in cfg:
        treat_key, ctrl_key = cfg["delta"]
        out["delta_by_group"] = delta_by_group(sm[treat_key], sm[ctrl_key], df=regions_tads)

    if "split" in cfg:
        (key,) = cfg["split"]
        out["split_by_group"] = split_by_group(sm[key], df=regions_tads)

    results_tads[name] = out

In [ ]:
regions_tads['sub'] = regions_tads['domain_score_wt3h'] - regions_tads['domain_score_wtnouv']
regions_tads['ratio'] = regions_tads['domain_score_wt3h'] / regions_tads['domain_score_wtnouv']

fig, axs = plt.subplots(1, 3, figsize=(12,4))
sns.kdeplot(data=regions_tads, x='sub', hue='Q', ax=axs[0])
axs[0].set_xlabel('Sub')
sns.kdeplot(data=regions_tads, x='ratio', hue='Q', ax=axs[1])
axs[1].set_xlabel('Ratio')
sns.kdeplot(data=regions_tads, x='domain_score_wt3h', hue='Q', ax=axs[2])
axs[2].set_xlabel('WT3h Domain Score')

In [ ]:
def znorm(arr, axis=0):
    mean = np.nanmean(arr, axis=axis, keepdims=True)
    std = np.nanstd(arr, axis=axis, keepdims=True)
    return (arr - mean) / std

fig, axs = plt.subplots(1,2, figsize=(20, 10), sharey=True)
for i, k in enumerate(['wt_nouv_ctcf', 'wt_3h_ctcf']):
    print(k)
    split_data = split_by_group(
    results_tads['ctcf_original']['raw'][k],
    regions_tads,
    group_col="Q",
)
    for q, arr in split_data.items():
        q = int(q)
        sns.lineplot(np.nanmean(arr, axis=0), ax=axs[i], label=f'{k} - Q{q}', color=list(reversed(plotting.COLORS))[q-1])

In [ ]:
def znorm(arr, axis=0):
    mean = np.nanmean(arr, axis=axis, keepdims=True)
    std = np.nanstd(arr, axis=axis, keepdims=True)
    return (arr - mean) / std

fig, axs = plt.subplots(1,2, figsize=(20, 10), sharey=True)
for i, k in enumerate(['wt_nouv_ds', 'wt_3h_ds']):
    print(k)
    split_data = split_by_group(
    results_tads['ds']['raw'][k],
    regions_tads,
    group_col="Q",
)
    
    for q, arr in split_data.items():
        q = int(q)
        if q in [2,3]:
            continue
        sns.lineplot(np.nanmean(arr, axis=0), ax=axs[i], label=f'{k} - Q{q}', color=list(reversed(plotting.COLORS))[q-1])

In [ ]:
def znorm(arr, axis=0):
    mean = np.nanmean(arr, axis=axis, keepdims=True)
    std = np.nanstd(arr, axis=axis, keepdims=True)
    return (arr - mean) / std

fig, axs = plt.subplots(1,3, figsize=(20, 10), sharey=True)
for i, k in enumerate(['wt_nouv_xpc', 'wt_1h_xpc', 'wt_3h_xpc']):
    print(k)
    split_data = split_by_group(
    results_tads['xpc']['raw'][k],
    regions_tads,
    group_col='Q',
)
    
    for q, arr in split_data.items():
        q = int(q)
        if q in [2,3]:
            continue
        sns.lineplot(np.nanmean(arr, axis=0), ax=axs[i], label=f'{k} - Q{q}', color=list(reversed(plotting.COLORS))[q-1])

In [ ]:
def znorm(arr, axis=0):
    mean = np.nanmean(arr, axis=axis, keepdims=True)
    std = np.nanstd(arr, axis=axis, keepdims=True)
    return (arr - mean) / std

fig, axs = plt.subplots(1,2, figsize=(12, 6), sharey=True)
for i, k in enumerate(['wt_nouv_atac', 'wt_3h_atac']):
    print(k)
    split_data = split_by_group(
    results_loops['atac']['raw'][k],
    results_loops,
    group_col='Q',
)
    
    for q, arr in split_data.items():
        q = int(q)
        if q in [2,3]:
            continue
        sns.lineplot(np.nanmean(arr, axis=0), ax=axs[i], label=f'{k} - Q{q}', color=list(reversed(plotting.COLORS))[q-1])

In [ ]:
ash1l_al_group_tads = split_array_by_group(
    regions_tads,
    gaussian_smooth(results_tads["ash1l_al"]["raw"]['wt_nouv'], sigma=None),
    group_col="Q",
    groups=(1,2,3,4)
)

ash1l_nouv_normed_tads = results_tads["ash1l_al"]["raw"]['wt_nouv'] - results_tads["ash1l_ev"]["raw"]['wt_nouv']
ash1l_3h_normed_tads = results_tads["ash1l_al"]["raw"]['wt_3h'] - results_tads["ash1l_ev"]["raw"]['wt_3h']
ash1l_delta_by_group_tads = groupwise_delta(
    regions_tads,
    gaussian_smooth(ash1l_3h_normed_tads, sigma=sigma),
    gaussian_smooth(ash1l_nouv_normed_tads, sigma=sigma),
    group_col="Q",
    groups=(1,2,3,4)
)

ash1l_al_group_loops = split_array_by_group(
    regions_loops,
    gaussian_smooth(results_loops["ash1l_al"]["raw"]['wt_nouv'], sigma=None),
    group_col="Q",
    groups=(1,2,3,4)
)

ash1l_nouv_normed_loops = results_loops["ash1l_al"]["raw"]['wt_nouv'] - results_loops["ash1l_ev"]["raw"]['wt_nouv']
ash1l_3h_normed_loops = results_loops["ash1l_al"]["raw"]['wt_3h'] - results_loops["ash1l_ev"]["raw"]['wt_3h']
ash1l_delta_by_group_loops = groupwise_delta(
    regions_loops,
    gaussian_smooth(ash1l_3h_normed_loops, sigma=sigma),
    gaussian_smooth(ash1l_nouv_normed_loops, sigma=sigma),
    group_col="Q",
    groups=(1,2,3,4)
)

In [ ]:
fig, axs = plt.subplots(9, 2, figsize=(36, 14), sharex='col')
q_colors = list(reversed(plotting.COLORS))

def add_vertical_lines(ax, n_main=100, n_flank=10):
    ax.axvline(n_flank, color='gray', linestyle='--', linewidth=1)
    ax.axvline(n_flank + n_main, color='gray', linestyle='--', linewidth=1)

def set_xaxis_labels(ax, n_main=100, n_flank=10, prefix='TAD'):
    xticks = [n_flank, n_flank + n_main//2, n_flank + n_main]
    xtick_labels = [f'{prefix} start', f'{prefix} center', f'{prefix} end']
    ax.set_xticks(xticks)
    ax.set_xticklabels(xtick_labels, rotation=45)

def plot_group_lines(ax, arr_by_group, label_prefix="Q", colors=None):
    for q in range(1, 5):
        mean_data = np.nanmean(arr_by_group[q], axis=0)
        sns.lineplot(
            x=np.arange(len(mean_data)),
            y=mean_data,
            label=f'{label_prefix}{q}',
            color=colors[q-1] if colors is not None else None,
            ax=ax,
            linestyle='-'  # ALWAYS solid for data
        )

# -----------------------
# Row map (0..8)
# 0: delta CTCF
# 1: Damage-seq (no UV) split
# 2: H3K4me3 3h (no UV) split
# 3: H3K4me3 1h (no UV) split
# 4: XPC 3h delta
# 5: XPC 1h delta
# 6: ASH1L (AL_SUB_EV) delta
# 7: Damage-seq (no UV - 3h) delta
# 8: ATAC-seq (no UV) split
# -----------------------

# ===== LEFT COL: TADs =====

# (0) delta ctcf tads
plot_group_lines(axs[0, 0], results_tads["ctcf"]["delta_by_group"], colors=q_colors)
axs[0, 0].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[0, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[0, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[0, 0].set_ylabel('Delta CTCF\n(3h - no UV)')

# (1) damage seq wt no uv tads (split)
plot_group_lines(axs[1, 0], results_tads["ds"]["split_by_group"], colors=q_colors)
add_vertical_lines(axs[1, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[1, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[1, 0].set_ylabel('Damage-seq\n(no UV)')

# (2) H3K4me3 3h tads (split)
plot_group_lines(axs[2, 0], results_tads["k4me3_3h"]["split_by_group"], colors=q_colors)
add_vertical_lines(axs[2, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[2, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[2, 0].set_ylabel('H3K4me3\n(3h - no UV)')

# (3) H3K4me3 1h tads (split)  (solid)
plot_group_lines(axs[3, 0], results_tads["k4me3_1h"]["split_by_group"], colors=q_colors)
add_vertical_lines(axs[3, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[3, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[3, 0].set_ylabel('H3K4me3\n(1h - no UV)')

# (4) XPC 3h delta tads
plot_group_lines(axs[4, 0], results_tads["xpc_3h"]["delta_by_group"], colors=q_colors)
axs[4, 0].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[4, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[4, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[4, 0].set_ylabel('XPC\n(3h - no UV)')

# (5) XPC 1h delta tads  (solid)
plot_group_lines(axs[5, 0], results_tads["xpc_1h"]["delta_by_group"], colors=q_colors)
axs[5, 0].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[5, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[5, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[5, 0].set_ylabel('XPC\n(1h - no UV)')

# (6) ASH1L (AL_SUB_EV) delta tads
plot_group_lines(axs[6, 0], results_tads["ash1l_al_sub_ev"]["delta_by_group"], colors=q_colors)
add_vertical_lines(axs[6, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[6, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[6, 0].set_ylabel('ASH1L (AL_SUB_EV)\n(3h - no UV)')

# (7) repair tads (ds delta)
plot_group_lines(axs[7, 0], results_tads["ds"]["delta_by_group"], colors=q_colors)
axs[7, 0].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[7, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[7, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[7, 0].set_ylabel('Damage-seq\n(no UV - 3h)')

# (8) atac wt no uv tads (split)
plot_group_lines(axs[8, 0], results_tads["atac"]["split_by_group"], colors=q_colors)
add_vertical_lines(axs[8, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[8, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[8, 0].set_ylabel('ATAC-seq\n(no UV)')


# ===== RIGHT COL: Loops =====

# (0) delta ctcf loops
plot_group_lines(axs[0, 1], results_loops["ctcf"]["delta_by_group"], colors=q_colors)
axs[0, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[0, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[0, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[0, 1].set_ylabel('Delta CTCF\n(3h - no UV)')

# (1) damage seq wt no uv loops (split)
plot_group_lines(axs[1, 1], results_loops["ds"]["split_by_group"], colors=q_colors)
add_vertical_lines(axs[1, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[1, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[1, 1].set_ylabel('Damage-seq\n(no UV)')

# (2) H3K4me3 3h loops (split)
plot_group_lines(axs[2, 1], results_loops["k4me3_3h"]["split_by_group"], colors=q_colors)
add_vertical_lines(axs[2, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[2, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[2, 1].set_ylabel('H3K4me3\n(3h - no UV)')

# (3) H3K4me3 1h loops (split) (solid)
plot_group_lines(axs[3, 1], results_loops["k4me3_1h"]["split_by_group"], colors=q_colors)
add_vertical_lines(axs[3, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[3, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[3, 1].set_ylabel('H3K4me3\n(1h - no UV)')

# (4) XPC 3h delta loops
plot_group_lines(axs[4, 1], results_loops["xpc_3h"]["delta_by_group"], colors=q_colors)
axs[4, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[4, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[4, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[4, 1].set_ylabel('XPC\n(3h - no UV)')

# (5) XPC 1h delta loops (solid)
plot_group_lines(axs[5, 1], results_loops["xpc_1h"]["delta_by_group"], colors=q_colors)
axs[5, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[5, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[5, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[5, 1].set_ylabel('XPC\n(1h - no UV)')

# (6) ASH1L (AL_SUB_EV) delta loops
plot_group_lines(axs[6, 1], results_loops["ash1l_al_sub_ev"]["delta_by_group"], colors=q_colors)
add_vertical_lines(axs[6, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[6, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[6, 1].set_ylabel('ASH1L (AL_SUB_EV)\n(3h - no UV)')

# (7) repair loops (ds delta)
plot_group_lines(axs[7, 1], results_loops["ds"]["delta_by_group"], colors=q_colors)
axs[7, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[7, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[7, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[7, 1].set_ylabel('Damage-seq\n(no UV - 3h)')

# (8) atac wt no uv loops (split)
plot_group_lines(axs[8, 1], results_loops["atac"]["split_by_group"], colors=q_colors)
add_vertical_lines(axs[8, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[8, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[8, 1].set_ylabel('ATAC-seq\n(no UV)')

# legends: keep only top row legends per column
for ax in axs.flatten():
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()

axs[0, 0].legend(title='Groups (Q1–Q4)', loc='best')
axs[0, 1].legend(title='Groups (Q1–Q4)', loc='best')

for ax in axs.flatten():
    plotting.despine(ax)

plt.tight_layout()
# fig.savefig(f'/home/carlos/Clone/ggner-3d/figs/fig2/scelad_tads_loops_stackups_Q.svg')

In [ ]:
fig, axs = plt.subplots(9, 2, figsize=(24, 18), sharex='col')

def add_vertical_lines(ax, n_main=100, n_flank=10):
    ax.axvline(n_flank, color='gray', linestyle='--', linewidth=1)
    ax.axvline(n_flank + n_main, color='gray', linestyle='--', linewidth=1)

def set_xaxis_labels(ax, n_main=100, n_flank=10, bin_size=bin_bp, prefix='TAD'):
    xticks = [0, n_flank, n_flank + n_main//2, n_flank + n_main, n_flank + n_main + n_flank]
    xtick_labels = [
        f'-{(n_flank * bin_size)//1000}kb',
        f'{prefix} start',
        f'{prefix} center',
        f'{prefix} end',
        f'+{(n_flank * bin_size)//1000}kb'
    ]
    ax.set_xticks(xticks)
    ax.set_xticklabels(xtick_labels, rotation=45)

def plot_mean(ax, arr, label=None, color=None, linestyle='-'):
    mean_data = np.nanmean(arr, axis=0)
    color = color if color is not None else 'blue'
    sns.lineplot(x=np.arange(len(mean_data)), y=mean_data, ax=ax, label=label, color=color, linestyle=linestyle)  # solid by default

# -----------------------
# Row map (0..8)
# 0: delta CTCF
# 1: Damage-seq (no UV)
# 2: H3K4me3 3h (no UV diff)
# 3: H3K4me3 1h (no UV diff)
# 4: XPC 3h delta
# 5: XPC 1h delta
# 6: ASH1L (AL_SUB_EV) delta
# 7: Damage-seq repair (no UV - 3h)
# 8: ATAC-seq (no UV)
# -----------------------

# ===== LEFT COL: TADs =====

# (0) delta CTCF tads
plot_mean(
    axs[0, 0],
    results_tads["ctcf"]['smoothed']['wt_3h_ctcf'] - results_tads["ctcf"]['smoothed']['wt_nouv_ctcf'],
    label='mean'
)
axs[0, 0].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[0, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[0, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[0, 0].set_ylabel('Delta CTCF\n(3h - no UV)')

# (1) Damage-seq no UV tads
plot_mean(axs[1, 0], results_tads["ds"]['smoothed']['wt_nouv_ds'], label='mean')
add_vertical_lines(axs[1, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[1, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[1, 0].set_ylabel('Damage-seq\n(no UV)')

# (2) H3K4me3 3h tads
plot_mean(axs[2, 0], results_tads["k4me3_3h"]['smoothed']['3h_nouv_diff'], label='mean')
axs[2, 0].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[2, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[2, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[2, 0].set_ylabel('H3K4me3\n(3h - no UV)')

# (3) H3K4me3 1h tads
plot_mean(axs[3, 0], results_tads["k4me3_1h"]['smoothed']['1h_nouv_diff'], label='mean')
axs[3, 0].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[3, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[3, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[3, 0].set_ylabel('H3K4me3\n(1h - no UV)')

# (4) XPC 3h tads
plot_mean(
    axs[4, 0],
    results_tads["xpc_3h"]['smoothed']['wt_3h_xpc'] - results_tads["xpc_3h"]['smoothed']['wt_nouv_xpc'],
    label='mean'
)
axs[4, 0].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[4, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[4, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[4, 0].set_ylabel('XPC\n(3h - no UV)')

# (5) XPC 1h tads
plot_mean(
    axs[5, 0],
    results_tads["xpc_1h"]['smoothed']['wt_1h_xpc'] - results_tads["xpc_1h"]['smoothed']['wt_nouv_xpc'],
    label='mean'
)
axs[5, 0].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[5, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[5, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[5, 0].set_ylabel('XPC\n(1h - no UV)')

# (6) ASH1L (AL_SUB_EV) delta tads
plot_mean(
    axs[6, 0],
    results_tads["ash1l_al_sub_ev"]["smoothed"]["wt_3h"] - results_tads["ash1l_al_sub_ev"]["smoothed"]["wt_nouv"],
    label='mean'
)
add_vertical_lines(axs[6, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[6, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[6, 0].set_ylabel('ASH1L (AL_SUB_EV)\n(3h - no UV)')

# (7) Damage-seq repair tads (no UV - 3h)
plot_mean(
    axs[7, 0],
    results_tads["ds"]['smoothed']['wt_nouv_ds'] - results_tads["ds"]['smoothed']['wt_3h_ds'],
    label='mean'
)
axs[7, 0].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[7, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[7, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[7, 0].set_ylabel('Damage-seq\n(no UV - 3h)')

# (8) ATAC-seq no UV tads
plot_mean(axs[8, 0], results_tads["atac"]["smoothed"]["wt_nouv_atac"], label='mean')
add_vertical_lines(axs[8, 0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[8, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[8, 0].set_ylabel('ATAC-seq\n(no UV)')


# ===== RIGHT COL: Loops =====

# (0) delta CTCF loops
plot_mean(
    axs[0, 1],
    results_loops["ctcf"]['smoothed']['wt_3h_ctcf'] - results_loops["ctcf"]['smoothed']['wt_nouv_ctcf'],
    label='mean'
)
axs[0, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[0, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[0, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[0, 1].set_ylabel('Delta CTCF\n(3h - no UV)')

# (1) Damage-seq no UV loops
plot_mean(axs[1, 1], results_loops["ds"]['smoothed']['wt_nouv_ds'], label='mean')
add_vertical_lines(axs[1, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[1, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[1, 1].set_ylabel('Damage-seq\n(no UV)')

# (2) H3K4me3 3h loops
plot_mean(axs[2, 1], results_loops["k4me3_3h"]['smoothed']['3h_nouv_diff'], label='mean')
axs[2, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[2, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[2, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[2, 1].set_ylabel('H3K4me3\n(3h - no UV)')

# (3) H3K4me3 1h loops
plot_mean(axs[3, 1], results_loops["k4me3_1h"]['smoothed']['1h_nouv_diff'], label='mean')
axs[3, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[3, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[3, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[3, 1].set_ylabel('H3K4me3\n(1h - no UV)')

# (4) XPC 3h loops
plot_mean(
    axs[4, 1],
    results_loops["xpc_3h"]['smoothed']['wt_3h_xpc'] - results_loops["xpc_3h"]['smoothed']['wt_nouv_xpc'],
    label='mean'
)
axs[4, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[4, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[4, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[4, 1].set_ylabel('XPC\n(3h - no UV)')

# (5) XPC 1h loops
plot_mean(
    axs[5, 1],
    results_loops["xpc_1h"]['smoothed']['wt_1h_xpc'] - results_loops["xpc_1h"]['smoothed']['wt_nouv_xpc'],
    label='mean'
)
axs[5, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[5, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[5, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[5, 1].set_ylabel('XPC\n(1h - no UV)')

# (6) ASH1L (AL_SUB_EV) delta loops
plot_mean(
    axs[6, 1],
    results_loops["ash1l_al_sub_ev"]["smoothed"]["wt_3h"] - results_loops["ash1l_al_sub_ev"]["smoothed"]["wt_nouv"],
    label='mean'
)
add_vertical_lines(axs[6, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[6, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[6, 1].set_ylabel('ASH1L (AL_SUB_EV)\n(3h - no UV)')

# (7) Damage-seq repair loops (no UV - 3h)
plot_mean(
    axs[7, 1],
    results_loops["ds"]['smoothed']['wt_nouv_ds'] - results_loops["ds"]['smoothed']['wt_3h_ds'],
    label='mean'
)
axs[7, 1].axhline(0, color='gray', linestyle='--', linewidth=1)
add_vertical_lines(axs[7, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[7, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[7, 1].set_ylabel('Damage-seq\n(no UV - 3h)')

# (8) ATAC-seq no UV loops
plot_mean(axs[8, 1], results_loops["atac"]["smoothed"]["wt_nouv_atac"], label='mean')
add_vertical_lines(axs[8, 1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[8, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[8, 1].set_ylabel('ATAC-seq\n(no UV)')

# cosmetics
for ax in axs.flatten():
    plotting.despine(ax)
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()

plt.tight_layout()


fig.savefig(f'/home/carlos/Clone/ggner-3d/figs/fig2/scelad_tads_loops_stackups.svg')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(18, 6))
plot_mean(axs[0], results_tads["ash1l_al_sub_ev"]["raw"]["wt_nouv"], label='ASH1L (AL - EV) noUV', color='blue', linestyle='-')
plot_mean(axs[1], results_loops["ash1l_al_sub_ev"]["raw"]["wt_nouv"], label='ASH1L (AL - EV) noUV', color='blue', linestyle='-')
plot_mean(axs[0], results_tads["ash1l_al_sub_ev"]["raw"]["wt_3h"], label='ASH1L (AL - EV) 3h', linestyle='--')
plot_mean(axs[1], results_loops["ash1l_al_sub_ev"]["raw"]["wt_3h"], label='ASH1L (AL - EV) 3h', linestyle='--')

# add vertical lines for boundaries and anchors
add_vertical_lines(axs[0], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[0], n_main=n_main, n_flank=n_flank, prefix='TAD')
axs[0].set_ylabel('ASH1L (AL - EV)')

add_vertical_lines(axs[1], n_main=n_main, n_flank=n_flank)
set_xaxis_labels(axs[1], n_main=n_main, n_flank=n_flank, prefix='Loop')
axs[1].set_ylabel('ASH1L (AL - EV)')

for ax in axs.flatten():
    plotting.despine(ax)
    ax.axhline(0, color='gray', linestyle='--', linewidth=1)
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()
axs[0].legend(title='TADs', loc='best')
axs[1].legend(title='Loops', loc='best')
plt.tight_layout()

fig.savefig(f'/home/carlos/Clone/ggner-3d/figs/fig2/scelad_ash1l_al_ev_tads_loops_stackups_timepointed.svg')

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(18, 6))
ash1l_split_wtnouv_tads = split_by_group(
    results_tads["ash1l_al_sub_ev"]["raw"]["wt_nouv"],
    df=regions_tads,
)
ash1l_split_wt3h_tads = split_by_group(
    results_tads["ash1l_al_sub_ev"]["raw"]["wt_3h"],
    df=regions_tads,
)
ash1l_split_wtnouv_loops = split_by_group(
    results_loops["ash1l_al_sub_ev"]["raw"]["wt_nouv"],
    df=regions_loops,
)
ash1l_split_wt3h_loops = split_by_group(
    results_loops["ash1l_al_sub_ev"]["raw"]["wt_3h"],
    df=regions_loops,
)

for q in range(1, 5):
    plot_mean(axs[0, 0], ash1l_split_wtnouv_tads[q], label=f'Q{q}', color=q_colors[q-1])
    # set y label
    axs[0, 0].set_ylabel('ASH1L (AL - EV)\nno UV - TADs')
    plot_mean(axs[1, 0], ash1l_split_wt3h_tads[q], label=f'Q{q}', color=q_colors[q-1])
    axs[1, 0].set_ylabel('ASH1L (AL - EV)\n3h - TADs')
    plot_mean(axs[0, 1], ash1l_split_wtnouv_loops[q], label=f'Q{q}', color=q_colors[q-1])
    axs[0, 1].set_ylabel('ASH1L (AL - EV)\nno UV - Loops')
    plot_mean(axs[1, 1], ash1l_split_wt3h_loops[q], label=f'Q{q}', color=q_colors[q-1])
    axs[1, 1].set_ylabel('ASH1L (AL - EV)\n3h - Loops')

for q in [1,4]:
    plot_mean(axs[2, 0], ash1l_split_wtnouv_tads[q], label=f'no UV - Q{q}', color=q_colors[q-1])
    plot_mean(axs[2, 0], ash1l_split_wt3h_tads[q], label=f'3h - Q{q}', color=q_colors[q-1], linestyle='--')
    axs[2, 0].set_ylabel('ASH1L (AL - EV)\nTADs')

    plot_mean(axs[2, 1], ash1l_split_wtnouv_loops[q], label=f'no UV - Q{q}', color=q_colors[q-1])
    plot_mean(axs[2, 1], ash1l_split_wt3h_loops[q], label=f'3h - Q{q}', color=q_colors[q-1], linestyle='--')
    axs[2, 1].set_ylabel('ASH1L (AL - EV)\nLoops')



for ax in axs.flatten():
    plotting.despine(ax)
    ax.axhline(0, color='gray', linestyle='--', linewidth=1)
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()


set_xaxis_labels(axs[2, 0], n_main=n_main, n_flank=n_flank, prefix='TAD')
set_xaxis_labels(axs[2, 1], n_main=n_main, n_flank=n_flank, prefix='Loop')

# add vertical lines for boundaries and anchors
add_vertical_lines(axs[0, 0], n_main=n_main, n_flank=n_flank)
add_vertical_lines(axs[1, 0], n_main=n_main, n_flank=n_flank)
add_vertical_lines(axs[0, 1], n_main=n_main, n_flank=n_flank)
add_vertical_lines(axs[1, 1], n_main=n_main, n_flank=n_flank)
add_vertical_lines(axs[2, 0], n_main=n_main, n_flank=n_flank)
add_vertical_lines(axs[2, 1], n_main=n_main, n_flank=n_flank)

axs[0, 0].legend(title='TADs - Groups (Q1-Q4)', loc='best')
axs[0, 1].legend(title='Loops - Groups (Q1-Q4)', loc='best')
# add legend to 2,0 and 2,1
axs[2, 0].legend(title='TADs - Groups (Q1, Q4)', loc='best')
axs[2, 1].legend(title='Loops - Groups (Q1, Q4)', loc='best')

plt.tight_layout()

fig.savefig(f'/home/carlos/Clone/ggner-3d/figs/fig2/scelad_ash1l_al_ev_tads_loops_stackups_timepointed_byQ.svg')

### split boundaries

In [ ]:
from ggner_3d.plotting import plot_flanks_start_end_stackup

In [ ]:
ds_dict_obs = {
    'wt_nouv_ds': ['/home/carlos/oldies/ner_collab/damageseq/obs0_rpm.1000bp.bw'],
    'wt_3h_ds': ['/home/carlos/oldies/ner_collab/damageseq/obs3_rpm.1000bp.bw'],
}

ds_dict_exp = {
    'wt_nouv_ds': ['/home/carlos/oldies/ner_collab/damageseq/exp0_rpm.1000bp.bw'],
    'wt_3h_ds': ['/home/carlos/oldies/ner_collab/damageseq/exp3_rpm.1000bp.bw'],
}

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# --- Figure / axes ---
fig, axs = plt.subplots(
    2, 2,
    figsize=(12, 6),
    sharey="row",
    gridspec_kw={"wspace": 0.2, "hspace": 0.3},
)

plotting.update_rcparams()

# --- Colors / legend handles (proxy legend; works even if your plotting function returns nothing) ---
c_3h = plotting.COLORS[1]
c_nouv = plotting.COLORS[0]

legend_handles = [
    Line2D([0], [0], color=c_3h, lw=1.5, label="WT 3h"),
    Line2D([0], [0], color=c_nouv, lw=1.5, label="WT nouv"),
]

# --- TADs: Observed (top-left) ---
plot_flanks_start_end_stackup(
    tad_str, ds_dict_obs["wt_3h_ds"],
    flank_bp=500_000, bins=100, palette=[c_3h],
    boundary_mode="concat", agg_across_regions="mean",
    ax=axs[0, 0], shade="sem", znorm="region",
    color_alpha=0.1, vline_alpha=0, lw=0.9,
    flip_3prime=False, shade_alpha=0.15, site_type="Boundary"
)
plot_flanks_start_end_stackup(
    tad_str, ds_dict_obs["wt_nouv_ds"],
    flank_bp=500_000, bins=100, palette=[c_nouv],
    boundary_mode="concat", agg_across_regions="mean",
    ax=axs[0, 0], shade="sem", znorm="region",
    color_alpha=0.1, vline_alpha=0, lw=0.9,
    flip_3prime=False, shade_alpha=0.15, site_type="Boundary"
)

# --- TADs: Expected (top-right) ---
plot_flanks_start_end_stackup(
    tad_str, ds_dict_exp["wt_3h_ds"],
    flank_bp=500_000, bins=100, palette=[c_3h],
    boundary_mode="concat", agg_across_regions="mean",
    ax=axs[0, 1], shade="sem", znorm="region",
    color_alpha=0.1, vline_alpha=0, lw=0.9,
    flip_3prime=False, shade_alpha=0.15, site_type="Boundary"
)
plot_flanks_start_end_stackup(
    tad_str, ds_dict_exp["wt_nouv_ds"],
    flank_bp=500_000, bins=100, palette=[c_nouv],
    boundary_mode="concat", agg_across_regions="mean",
    ax=axs[0, 1], shade="sem", znorm="region",
    color_alpha=0.1, vline_alpha=0, lw=0.9,
    flip_3prime=False, shade_alpha=0.15, site_type="Boundary"
)

# --- Loops: Observed (bottom-left) ---
plot_flanks_start_end_stackup(
    dots, ds_dict_obs["wt_3h_ds"],
    flank_bp=200_000, bins=100, palette=[c_3h],
    boundary_mode="concat", agg_across_regions="mean",
    ax=axs[1, 0], shade="sem", znorm="region",
    color_alpha=0.1, vline_alpha=0, lw=0.9,
    flip_3prime=False, shade_alpha=0.15, site_type="Anchor"
)
plot_flanks_start_end_stackup(
    dots, ds_dict_obs["wt_nouv_ds"],
    flank_bp=200_000, bins=100, palette=[c_nouv],
    boundary_mode="concat", agg_across_regions="mean",
    ax=axs[1, 0], shade="sem", znorm="region",
    color_alpha=0.1, vline_alpha=0, lw=0.9,
    flip_3prime=False, shade_alpha=0.15, site_type="Anchor"
)

# --- Loops: Expected (bottom-right) ---
plot_flanks_start_end_stackup(
    dots, ds_dict_exp["wt_3h_ds"],
    flank_bp=200_000, bins=100, palette=[c_3h],
    boundary_mode="concat", agg_across_regions="mean",
    ax=axs[1, 1], shade="sem", znorm="region",
    color_alpha=0.1, vline_alpha=0, lw=0.9,
    flip_3prime=False, shade_alpha=0.15, site_type="Anchor"
)
plot_flanks_start_end_stackup(
    dots, ds_dict_exp["wt_nouv_ds"],
    flank_bp=200_000, bins=100, palette=[c_nouv],
    boundary_mode="concat", agg_across_regions="mean",
    ax=axs[1, 1], shade="sem", znorm="region",
    color_alpha=0.1, vline_alpha=0, lw=0.9,
    flip_3prime=False, shade_alpha=0.15, site_type="Anchor"
)

# --- Titles / labels ---
axs[0, 0].set_title("Damage-seq Observed")
axs[0, 1].set_title("Damage-seq Expected")
axs[1, 0].set_title("")
axs[1, 1].set_title("")

axs[0, 0].set_ylabel("TADs\nStandardized RPM")
axs[1, 0].set_ylabel("Loops\nStandardized RPM")
axs[0, 1].set_ylabel("")
axs[1, 1].set_ylabel("")

n_tads = len(tad_str)
n_loops = len(dots)

# Put counts inside the relevant panels
axs[0,0].text(0.02, 0.98, f"n = {n_tads}", transform=axs[0,0].transAxes,
             va="top", ha="left")
axs[1,0].text(0.02, 0.98, f"n = {n_loops}", transform=axs[1,0].transAxes,
             va="top", ha="left")

# --- Axis cosmetics ---
for ax in axs.flatten():
    plotting.despine(ax)
    ax.set_xlabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=15)

# --- One legend for the whole figure ---
fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=2,
    frameon=False,
    title="Condition",
    bbox_to_anchor=(0.5, 1.02),
)

# Leave room for the top legend
fig.tight_layout(rect=[0, 0, 1, 0.95])

# --- Save ---
fig.savefig(
    "/home/carlos/Clone/ggner-3d/figs/fig2/ds_observed_expected_stackups.svg",
)


In [ ]:
ctcf_wt_dict = {
'wt_nouv_ctcf': [
    '/home/carlos/oldies/ner_collab/ctcf/bigwigs_rpgc/WT_NoUV_REP1.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/ctcf/bigwigs_rpgc/WT_NoUV_REP2.mLb.clN.rpgc.bw'
],
'wt_3h_ctcf': [
    '/home/carlos/oldies/ner_collab/ctcf/bigwigs_rpgc/WT_3h_REP1.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/ctcf/bigwigs_rpgc/WT_3h_REP2.mLb.clN.rpgc.bw'
]
}

# ds_wt_dict = {
# 'wt_nouv_ds': [
#     '/home/carlos/oldies/ner_collab/damageseq/U2OS_CPD_0h_DS_hg38_pe_sortedbyCoordinates.rpgc.bw'
# ],
# 'wt_3h_ds': [
#     '/home/carlos/oldies/ner_collab/damageseq/U2OS_CPD_3h_DS_hg38_pe_sortedbyCoordinates.rpgc.bw'
# ]
# }

k4me3_wt_dict = {
'3h_nouv_diff': ['/home/carlos/oldies/ner_collab/k4m3-ash-1h3h/3h.ASH1L_specific.delta_mean.bw'],
'1h_nouv_diff': ['/home/carlos/oldies/ner_collab/k4m3-ash-1h3h/1h.ASH1L_specific.delta_mean.bw']
}

xpc_wt_dict = {
'wt_nouv_xpc': [
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_noUV_REP1.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_noUV_REP2.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_noUV_REP3.mLb.clN.rpgc.bw'
],
'wt_3h_xpc': [
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_3h_REP1.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_3h_REP2.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_3h_REP3.mLb.clN.rpgc.bw'
],
'wt_1h_xpc': [
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_1h_REP1.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_1h_REP2.mLb.clN.rpgc.bw',
    '/home/carlos/oldies/ner_collab/xpc_chipseq/bigwigs_rpgc/XPC_WT_1h_REP3.mLb.clN.rpgc.bw'
]
}

ash1l_wt_al_dict = {
    'wt_nouv': ['/home/carlos/oldies/ner_collab/ASH1L_RPGC/AL_noUV_1.rpgc.bw',
                '/home/carlos/oldies/ner_collab/ASH1L_RPGC/AL_noUV_2.rpgc.bw',
                '/home/carlos/oldies/ner_collab/ASH1L_RPGC/AL_noUV_3.rpgc.bw'],
    'wt_3h': ['/home/carlos/oldies/ner_collab/ASH1L_RPGC/AL_3H_1.rpgc.bw',
              '/home/carlos/oldies/ner_collab/ASH1L_RPGC/AL_3H_2.rpgc.bw',
              '/home/carlos/oldies/ner_collab/ASH1L_RPGC/AL_3H_3.rpgc.bw']
}

ash1l_wt_ev_dict = {
    'wt_nouv': ['/home/carlos/oldies/ner_collab/ASH1L_RPGC/EV_noUV_1.rpgc.bw',
                '/home/carlos/oldies/ner_collab/ASH1L_RPGC/EV_noUV_2.rpgc.bw',
                '/home/carlos/oldies/ner_collab/ASH1L_RPGC/EV_noUV_3.rpgc.bw'],
    'wt_3h': ['/home/carlos/oldies/ner_collab/ASH1L_RPGC/EV_3H_1.rpgc.bw',
              '/home/carlos/oldies/ner_collab/ASH1L_RPGC/EV_3H_2.rpgc.bw',
              '/home/carlos/oldies/ner_collab/ASH1L_RPGC/EV_3H_3.rpgc.bw']
}

ash1l_wt_al_sub_ev_dict = {
    'wt_nouv': ['/home/carlos/oldies/ner_collab/ASH1L_RPGC/ALminusEV_noUV.bw'],
    'wt_3h': ['/home/carlos/oldies/ner_collab/ASH1L_RPGC/ALminusEV_3H.bw']
}

atac_wt_dict = {
    'wt_nouv_atac': [
        '/home/carlos/oldies/ner_collab/atac/WT_noUV_1.fc.bigwig',
        '/home/carlos/oldies/ner_collab/atac/WT_noUV_2.fc.bigwig'
    ],
    'wt_3h_atac': [
        '/home/carlos/oldies/ner_collab/atac/WT_3H_1.fc.bigwig',
        '/home/carlos/oldies/ner_collab/atac/WT_3H_2.fc.bigwig'
    ]
}

In [ ]:
import importlib
from ggner_3d import plotting 
importlib.reload(plotting)
from ggner_3d.plotting import plot_flanks_start_end_stackup

In [ ]:
aspect = (4,3)
aspect_multiplier = 3.75
asp = (aspect[0]*aspect_multiplier, aspect[1]*aspect_multiplier)
fig, axs = plt.subplots(6, 2, figsize=(asp[0], asp[1]), gridspec_kw={'wspace':0.2, 'hspace':0.3})

# -----------------------
# Window / bin parameters
# -----------------------
tad_flank = 500_000
loop_flank = 200_000

tad_nbins  = 150
loop_nbins = 80
print(f"TAD nbins: {tad_nbins}, Loop nbins: {loop_nbins}")

tad_gap_bp  = 200_000
loop_gap_bp = 100_000 # *_gap_bp is the region not visualized in the center of the stackup. just technical thing to viz in the same axis.
summary_per_bin = 'mean'

znorm = 'region'

# -----------------------
# Colors
# -----------------------
c_3h = plotting.COLORS[1]
c_1h = plotting.COLORS[2]
c_nouv = plotting.COLORS[0]
c_single_line = plotting.COLORS[0]

# -----------------------
# Common plotting kwargs
# -----------------------
COMMON_STACKUP_KW = dict(
    boundary_mode="concat",
    agg_across_regions="mean",
    shade="sem",
    color_alpha=0.05,
    vline_alpha=0,
    lw=0.9,
    flip_3prime=False,
    shade_alpha=0.10,
    summary_per_bin=summary_per_bin
)

# Panel-type presets (TAD boundary vs loop anchor)
TAD_PRESET = dict(
    flank_bp=tad_flank,
    bins=tad_nbins,
    site_type="Boundary",
    gap_bp=tad_gap_bp,
)

LOOP_PRESET = dict(
    flank_bp=loop_flank,
    bins=loop_nbins,
    site_type="Anchor",
    gap_bp=loop_gap_bp,
)

# Assay-specific presets (znorm differs)
CTCF_PRESET = dict(znorm=znorm)
DS_PRESET   = dict(znorm=znorm)
H3K4ME3_PRESET = dict(znorm=znorm)
XPC_PRESET = dict(znorm=znorm)
ASH1L_PRESET = dict(znorm=znorm)
ATAC_PRESET = dict(znorm=znorm)

# axis titles
y_axis_titles = [
    "CTCF\nMean RPGC Signal",
    "Damage-seq\nMean RPM Signal",
    "Δ H3K4me3\n(vs. WT noUV)",
    "XPC\nMean RPGC Signal",
    "ASH1L\nMean RPGC Signal",
    "ATAC-seq\nFold Change Signal",
]

# Convenience helper so calls stay short
def stackup(ax, regions, signal, palette, preset, assay):
    plot_flanks_start_end_stackup(
        regions, signal,
        ax=ax,
        palette=palette,
        **COMMON_STACKUP_KW,
        **preset,
        **assay,
    )

######
# CTCF - RPGC signal
######
stackup(axs[0, 0], tad_str, ctcf_wt_dict["wt_3h_ctcf"],   [c_3h],   TAD_PRESET,  CTCF_PRESET)
stackup(axs[0, 1], dots,    ctcf_wt_dict["wt_3h_ctcf"],   [c_3h],   LOOP_PRESET, CTCF_PRESET)

stackup(axs[0, 0], tad_str, ctcf_wt_dict["wt_nouv_ctcf"], [c_nouv], TAD_PRESET,  CTCF_PRESET)
stackup(axs[0, 1], dots,    ctcf_wt_dict["wt_nouv_ctcf"], [c_nouv], LOOP_PRESET, CTCF_PRESET)

# #####
# Damage-seq RPM Signal
# #####
stackup(axs[1, 0], tad_str, ds_dict_obs["wt_3h_ds"],   [c_3h],   TAD_PRESET,  DS_PRESET)
stackup(axs[1, 1], dots,    ds_dict_obs["wt_3h_ds"],   [c_3h],   LOOP_PRESET, DS_PRESET)

stackup(axs[1, 0], tad_str, ds_dict_obs["wt_nouv_ds"], [c_nouv], TAD_PRESET,  DS_PRESET)
stackup(axs[1, 1], dots,    ds_dict_obs["wt_nouv_ds"], [c_nouv], LOOP_PRESET, DS_PRESET)

####
# H3K4me3 - Ash1L specific delta signal
#####

stackup(axs[2, 0], tad_str, k4me3_wt_dict["1h_nouv_diff"], [c_1h], TAD_PRESET, H3K4ME3_PRESET)
stackup(axs[2, 1], dots,    k4me3_wt_dict["1h_nouv_diff"], [c_1h], LOOP_PRESET, H3K4ME3_PRESET)

stackup(axs[2, 0], tad_str, k4me3_wt_dict["3h_nouv_diff"], [c_3h], TAD_PRESET, H3K4ME3_PRESET)
stackup(axs[2, 1], dots,    k4me3_wt_dict["3h_nouv_diff"], [c_3h], LOOP_PRESET, H3K4ME3_PRESET)

####
# XPC
# RPGC signal
####
stackup(axs[3, 0], tad_str, xpc_wt_dict["wt_3h_xpc"],   [c_3h],   TAD_PRESET,  XPC_PRESET)
stackup(axs[3, 1], dots,    xpc_wt_dict["wt_3h_xpc"],   [c_3h],   LOOP_PRESET, XPC_PRESET)

stackup(axs[3, 0], tad_str, xpc_wt_dict["wt_nouv_xpc"], [c_nouv], TAD_PRESET,  XPC_PRESET)
stackup(axs[3, 1], dots,    xpc_wt_dict["wt_nouv_xpc"], [c_nouv], LOOP_PRESET, XPC_PRESET)

stackup(axs[3, 0], tad_str, xpc_wt_dict["wt_1h_xpc"],   [c_1h],   TAD_PRESET,  XPC_PRESET)
stackup(axs[3, 1], dots,    xpc_wt_dict["wt_1h_xpc"],   [c_1h],   LOOP_PRESET, XPC_PRESET)

####
# ASH1L
####

stackup(axs[4, 0], tad_str, ash1l_wt_al_sub_ev_dict["wt_3h"],   [c_3h],   TAD_PRESET,  ASH1L_PRESET)
stackup(axs[4, 1], dots,    ash1l_wt_al_sub_ev_dict["wt_3h"],   [c_3h],   LOOP_PRESET, ASH1L_PRESET)

stackup(axs[4, 0], tad_str, ash1l_wt_al_sub_ev_dict["wt_nouv"], [c_nouv], TAD_PRESET,  ASH1L_PRESET)
stackup(axs[4, 1], dots,    ash1l_wt_al_sub_ev_dict["wt_nouv"], [c_nouv], LOOP_PRESET, ASH1L_PRESET)


####
# ATAC-seq
####

stackup(axs[5, 0], tad_str, atac_wt_dict["wt_nouv_atac"], [c_nouv], TAD_PRESET, ATAC_PRESET)
stackup(axs[5, 1], dots,    atac_wt_dict["wt_nouv_atac"], [c_nouv], LOOP_PRESET, ATAC_PRESET)

stackup(axs[5, 0], tad_str, atac_wt_dict["wt_3h_atac"],   [c_3h],   TAD_PRESET, ATAC_PRESET)
stackup(axs[5, 1], dots,    atac_wt_dict["wt_3h_atac"],   [c_3h],   LOOP_PRESET, ATAC_PRESET)


for i in range(6):
    axs[i, 0].set_ylabel(y_axis_titles[i])
    axs[i, 1].set_ylabel("")
    axs[i, 0].set_title("")
    axs[i, 1].set_title("")

# set global titles for columns
axs[0, 0].set_title(f"TADs (n={len(tad_str)}) Boundaries")
axs[0, 1].set_title(f"Loops (n={len(dots)}) Anchors")

# --- Axis cosmetics ---
for i, ax in enumerate(axs.flatten()):
    plotting.despine(ax)
    ax.set_xlabel("")

    if i in (10, 11):   # bottom row
        ax.tick_params(axis="x", labelbottom=True)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
    else:
        ax.tick_params(axis="x", labelbottom=False)

fig.tight_layout()
fig.savefig(f"/home/carlos/Clone/ggner-3d/figs/fig2/1_stackups_all_assays_boundaries_anchors_znorm{znorm}_tadnbins{tad_nbins}_loop_nbins{loop_nbins}_summary{summary_per_bin}.svg")

In [ ]:
fig, axs = plt.subplots(2,2, figsize=(16,8), sharex='col')

znorm_now = None
ASH1L_PRESET = dict(znorm=znorm_now)

stackup(axs[0, 0], tad_str, ash1l_wt_al_dict["wt_3h"],   [c_3h],   TAD_PRESET,  ASH1L_PRESET)
stackup(axs[0, 1], dots,    ash1l_wt_al_dict["wt_3h"],   [c_3h],   LOOP_PRESET, ASH1L_PRESET)

stackup(axs[0, 0], tad_str, ash1l_wt_al_dict["wt_nouv"], [c_nouv], TAD_PRESET,  ASH1L_PRESET)
stackup(axs[0, 1], dots,    ash1l_wt_al_dict["wt_nouv"], [c_nouv], LOOP_PRESET, ASH1L_PRESET)

stackup(axs[1, 0], tad_str, ash1l_wt_ev_dict["wt_3h"],   [c_3h],   TAD_PRESET,  ASH1L_PRESET)
stackup(axs[1, 1], dots,    ash1l_wt_ev_dict["wt_3h"],   [c_3h],   LOOP_PRESET, ASH1L_PRESET)

stackup(axs[1, 0], tad_str, ash1l_wt_ev_dict["wt_nouv"], [c_nouv], TAD_PRESET,  ASH1L_PRESET)
stackup(axs[1, 1], dots,    ash1l_wt_ev_dict["wt_nouv"], [c_nouv], LOOP_PRESET, ASH1L_PRESET)

# add titles to columns
axs[0, 0].set_title(f"TADs (n={len(tad_str)}) Boundaries")
axs[0, 1].set_title(f"Loops (n={len(dots)}) Anchors")

# add titles to rows
axs[0, 0].set_ylabel("ASH1L AL RPGC Signal")
axs[1, 0].set_ylabel("ASH1L EV RPGC Signal")

fig.savefig(f"/home/carlos/Clone/ggner-3d/figs/fig2/stackups_ash1l_al_ev_boundaries_anchors_znorm{znorm_now}_tadnbins{tad_nbins}_loop_nbins{loop_nbins}_summary{summary_per_bin}.svg")